# Publish <SCHEMANAME> to SAP BDC

Publishes all 12 Iceberg V3 tables from `TECHED_DA261.<SCHEMANAME>` as a data product to SAP BDC via the zerocopy connector at `DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE`.

**Tables included (12):**
`FORECAST_VS_MILESTONES_SAP`, `LEARNING_FORECAST_RAW_SAP`, `TRANSFORMATION_MILESTONES`, `VW_CDP_FEATURES`, `VW_LABELED_EMPLOYEES`, `VW_LEARNING_FEATURES`, `VW_LEARNING_MONTHLY_SERIES`, `VW_PERFORMANCE_FEATURES`, `VW_SKILLS_SIGNAL`, `VW_SUCCESSION_FEATURES`, `WORKFORCE_READINESS_SCORES_SAP`, `WORKFORCE_TRANSFORMATION_VELOCITY`

**Run order:** Cell 1 → Cell 2 → Cell 3

✍️ Replace all
`<SCHEMANAME>`: SCHEMA_<YOUR USERNAME>
`<SHARENAME>` : TECHED_DA261_SHARE_<YOUR USERNAME>
`<USERNAME>`: YOUR_USERNAME

In [ ]:
%%sql -r share_result
BEGIN
    -- Create share
    CREATE SHARE IF NOT EXISTS <SHARENAME>;

    -- Grant usage on database and schema
    GRANT USAGE ON DATABASE TECHED_DA261 TO SHARE <SHARENAME>;
    GRANT USAGE ON SCHEMA TECHED_DA261.<SCHEMANAME> TO SHARE <SHARENAME>;

    -- Grant SELECT on all 12 Iceberg tables
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.FORECAST_VS_MILESTONES_SAP TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.LEARNING_FORECAST_RAW_SAP TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.TRANSFORMATION_MILESTONES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_CDP_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LEARNING_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_LEARNING_MONTHLY_SERIES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_PERFORMANCE_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_SKILLS_SIGNAL TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.VW_SUCCESSION_FEATURES TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.WORKFORCE_READINESS_SCORES_SAP TO SHARE <SHARENAME>;
    GRANT SELECT ON TABLE TECHED_DA261.<SCHEMANAME>.WORKFORCE_TRANSFORMATION_VELOCITY TO SHARE <SHARENAME>;

    -- Attach share to connector
    ALTER ZEROCOPY CONNECTOR DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE
        ADD SHARE <SHARENAME>;

EXCEPTION
    WHEN OTHER THEN
        RETURN 'Share may already be associated: ' || SQLERRM;
END;

In [ ]:
import json

ord_metadata = json.dumps({
    "title": "<USERNAME> Workforce Transformation Velocity Intelligence",
    "shortDescription": "ML-predicted employee readiness tiers and 12-month department learning trajectory vs transformation milestones.",
    "description": "Combined Snowflake Cortex ML output derived from 9 SAP SuccessFactors data products shared via SAP Business Data Cloud. ML.CLASSIFICATION scores each employee into transformation readiness tiers."
})

def emp_key():
    return {
        "type": "cds.String", "length": 5000,
        "key": True, "notNull": True,
        "@PersonalData.fieldSemantics": {"#": "DataSubjectIDType"}
    }

def string():
    return {"type": "cds.String", "length": 5000}

def decimal(p, s):
    return {"type": "cds.Decimal", "precision": p, "scale": s}

def double():
    return {"type": "cds.Double"}

def timestamp():
    return {"type": "cds.Timestamp"}

csn = {
    "csnInteropEffective": "1.0",
    "$version": "2.0",
    "i18n": {},
    "meta": {
        "creator": "Snowflake CSN Interop Generator - Minimal",
        "flavor": "inferred"
    },
    "definitions": {
        "<SCHEMANAME>": {"kind": "context"},

        "<SCHEMANAME>.FORECAST_VS_MILESTONES_SAP": {
            "kind": "entity",
            "elements": {
                "DEPARTMENT": {**decimal(20, 0), "key": True, "notNull": True},
                "FORECAST_MONTH": timestamp(),
                "MILESTONE_NAME": {**string(), "key": True, "notNull": True},
                "MILESTONE_STATUS": string(),
                "PROJECTED_COMPLETIONS": double(),
                "REQUIRED_COMPLETIONS": decimal(2, 0),
                "LOWER_90": double(),
                "UPPER_90": double()
            }
        },

        "<SCHEMANAME>.LEARNING_FORECAST_RAW_SAP": {
            "kind": "entity",
            "elements": {
                "SERIES": {**decimal(20, 0), "key": True, "notNull": True},
                "TS": {**timestamp(), "key": True, "notNull": True},
                "FORECAST": double(),
                "LOWER_BOUND": double(),
                "UPPER_BOUND": double()
            }
        },

        "<SCHEMANAME>.TRANSFORMATION_MILESTONES": {
            "kind": "entity",
            "elements": {
                "DEPARTMENT": {**decimal(20, 0), "key": True, "notNull": True},
                "MILESTONE_DATE": timestamp(),
                "MILESTONE_NAME": {**string(), "key": True, "notNull": True},
                "REQUIRED_COMPLETIONS": decimal(2, 0)
            }
        },

        "<SCHEMANAME>.VW_CDP_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "CDP_MILESTONE_COMPLETION_RATE": decimal(22, 6),
                "ACTIVE_DEV_GOAL_COUNT": decimal(13, 0)
            }
        },

        "<SCHEMANAME>.VW_LABELED_EMPLOYEES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "DIGITAL_PROGRAM_COMPLETION_RATE": decimal(22, 6),
                "LEARNING_HOURS_LAST_6MO": decimal(38, 17),
                "LEARNING_VELOCITY_TREND": string(),
                "DAYS_SINCE_LAST_LEARNING": decimal(9, 0),
                "CDP_MILESTONE_COMPLETION_RATE": decimal(22, 6),
                "ACTIVE_DEV_GOAL_COUNT": decimal(13, 0),
                "READINESS_LEVEL_DELTA_12MO": double(),
                "HIGH_POTENTIAL_FLAG": decimal(1, 0),
                "LATEST_PERF_RATING": decimal(34, 17),
                "PERF_RATING_DELTA": decimal(35, 17),
                "GOAL_COMPLETION_RATE": decimal(22, 6),
                "SKILL_PROFILE_UPDATES_6MO": decimal(13, 0),
                "READINESS_TIER_LABEL": string()
            }
        },

        "<SCHEMANAME>.VW_LEARNING_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "DIGITAL_PROGRAM_COMPLETION_RATE": decimal(22, 6),
                "LEARNING_HOURS_LAST_6MO": decimal(38, 17),
                "HOURS_RECENT_3MO": decimal(38, 17),
                "HOURS_PRIOR_3MO": decimal(38, 17),
                "DAYS_SINCE_LAST_LEARNING": decimal(9, 0)
            }
        },

        "<SCHEMANAME>.VW_LEARNING_MONTHLY_SERIES": {
            "kind": "entity",
            "elements": {
                "DEPARTMENT": {**decimal(20, 0), "key": True, "notNull": True},
                "MONTH_DATE": {**timestamp(), "key": True, "notNull": True},
                "COMPLETIONS": decimal(18, 0),
                "ACTIVE_LEARNERS": decimal(18, 0),
                "TOTAL_CREDIT_HOURS": decimal(38, 17)
            }
        },

        "<SCHEMANAME>.VW_PERFORMANCE_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "LATEST_PERF_RATING": decimal(34, 17),
                "PERF_RATING_DELTA": decimal(35, 17),
                "GOAL_COMPLETION_RATE": decimal(22, 6)
            }
        },

        "<SCHEMANAME>.VW_SKILLS_SIGNAL": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "SKILL_PROFILE_UPDATES_6MO": decimal(13, 0)
            }
        },

        "<SCHEMANAME>.VW_SUCCESSION_FEATURES": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "CURRENT_READINESS_LEVEL": string(),
                "READINESS_LEVEL_DELTA_12MO": double(),
                "HIGH_POTENTIAL_FLAG": decimal(1, 0)
            }
        },

        "<SCHEMANAME>.WORKFORCE_READINESS_SCORES_SAP": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "DEPARTMENT": decimal(20, 0),
                "JOB_TITLE": string(),
                "READINESS_TIER": string(),
                "P_READY_NOW": double(),
                "P_FAST_TRACK": double(),
                "P_NEEDS_COACHING": double(),
                "P_NEEDS_PROGRAM": double(),
                "READINESS_VELOCITY_INDEX": double(),
                "PREDICTION": string()
            }
        },

        "<SCHEMANAME>.WORKFORCE_TRANSFORMATION_VELOCITY": {
            "kind": "entity",
            "@PersonalData.entitySemantics": "DataSubjectDetails",
            "elements": {
                "EMPLOYEE_ID": emp_key(),
                "DEPARTMENT": decimal(38, 0),
                "JOB_TITLE": string(),
                "READINESS_TIER": string(),
                "READINESS_VELOCITY_INDEX": double(),
                "DEPT_MILESTONE_STATUS": string(),
                "DEPT_PROJECTED_COMPLETIONS_12MO": double()
            }
        }
    }
}

csn_json = json.dumps(csn)
print(f"ORD metadata: {len(ord_metadata)} chars")
print(f"CSN document: {len(csn_json)} chars")
print(f"Entities: {sum(1 for v in csn['definitions'].values() if isinstance(v, dict) and v.get('kind') == 'entity')}")
print(f"Total columns: {sum(len(v.get('elements', {})) for v in csn['definitions'].values() if isinstance(v, dict) and v.get('kind') == 'entity')}")

In [ ]:
%%sql -r publish_result
SELECT SYSTEM$SAP_PUBLISH_DATA_PRODUCT(
    'DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE',
    '<SHARENAME>',
    '{{ord_metadata}}',
    '{{csn_json}}'
)